# Smart Meal Planner Agent
## Strands Agents + AgentCore Memory + Streaming on Amazon Bedrock AgentCore Runtime

---

### What You Will Build

A personal meal planner agent that:
- Searches real recipes using the free **TheMealDB API** (no API key needed)
- **Remembers** your dietary preferences across sessions using **AgentCore Memory**
- **Streams** responses token-by-token for a better user experience
- Runs serverlessly on **Amazon Bedrock AgentCore Runtime**

### Tutorial Details

| Detail | Value |
|:---|:---|
| Agent Framework | Strands Agents |
| LLM Model | **Amazon Nova 2 Lite** (via Amazon Bedrock) |
| Model ID | `global.amazon.nova-2-lite-v1:0` |
| External API | TheMealDB (free, no key required) |
| New Features | AgentCore Memory, Streaming |
| Complexity | Easy — Beginner Friendly |

### Architecture

```
 ┌─────────────────────────────────────────────────────────────┐
 │                  AgentCore Runtime (Cloud)                   │
 │                                                             │
 │   User Prompt                                               │
 │       │                                                     │
 │       ▼                                                     │
 │  ┌─────────────┐    ┌──────────────┐    ┌───────────────┐  │
 │  │  AgentCore  │───▶│ Strands Agent│───▶│ Nova 2 Lite   │  │
 │  │   Memory    │    │              │    │  (Bedrock)    │  │
 │  │ (recall     │◀───│  Tools:      │    └───────────────┘  │
 │  │  prefs)     │    │  search_     │                       │
 │  └─────────────┘    │  recipe()    │    ┌───────────────┐  │
 │                     │  get_meals   │───▶│  TheMealDB    │  │
 │  ┌─────────────┐    │  _by_cat()   │    │  (free API)   │  │
 │  │  AgentCore  │    │  calculator()│    └───────────────┘  │
 │  │   Memory    │    └──────────────┘                       │
 │  │ (store new  │                                           │
 │  │  prefs)     │          │ Streaming response             │
 │  └─────────────┘          ▼                               │
 │                      User sees tokens                       │
 │                      arriving live                          │
 └─────────────────────────────────────────────────────────────┘
```

### Sections
1. Install Dependencies
2. Build the Local Agent (no memory, no streaming)
3. The Memory Problem — why memory matters
4. Add AgentCore Memory — agent that remembers
5. Add Streaming — real-time token output
6. Deploy Everything to AgentCore Runtime
7. Cleanup

---
## Prerequisites

Before running this notebook, make sure you have:
- Python 3.10+
- AWS credentials configured (`aws configure`)
- Amazon Bedrock model access enabled for Claude Haiku 4.5
- Docker installed (for local container build, optional — CodeBuild is used by default)

---
## Section 1 — Install Dependencies

In [ ]:
!pip install -r requirements.txt --quiet
!pip install requests --quiet

---
## Section 2 — Build the Local Agent

We start by building and testing the agent **locally**, before deploying to the cloud.

### Why TheMealDB?
- Completely **free** — no API key required
- Real recipe data: ingredients, instructions, categories
- Simple REST API — great for learning
- URL: `https://www.themealdb.com/api/json/v1/1/`

### Agent Tools
| Tool | What it does |
|:---|:---|
| `search_recipe(dish)` | Fetches a full recipe by name (TheMealDB) |
| `get_meals_by_category(category)` | Lists meals by category: Vegetarian, Seafood, Pasta, Dessert... |
| `calculator` | Scales recipe quantities (reused from strands_tools) |

In [ ]:
%%writefile meal_planner.py
from strands import Agent, tool
from strands_tools import calculator
from strands.models import BedrockModel
import requests
import json
import argparse

# ─────────────────────────────────────────────────────────────
# TOOL 1: Search a recipe by dish name
# Uses TheMealDB — completely free, no API key needed
# ─────────────────────────────────────────────────────────────
@tool
def search_recipe(dish: str) -> str:
    """Search for a recipe by dish name using TheMealDB API"""
    url = f"https://www.themealdb.com/api/json/v1/1/search.php?s={dish}"
    try:
        resp = requests.get(url, timeout=10)
        data = resp.json()

        if not data.get('meals'):
            return f"No recipe found for '{dish}'. Try a different name."

        meal = data['meals'][0]

        # Build ingredients list from the 20 possible ingredient slots
        ingredients = []
        for i in range(1, 21):
            ing = meal.get(f'strIngredient{i}', '').strip()
            measure = meal.get(f'strMeasure{i}', '').strip()
            if ing:
                ingredients.append(f"  - {measure} {ing}".strip())

        return (
            f"Recipe: {meal['strMeal']}\n"
            f"Category: {meal['strCategory']}  |  Cuisine: {meal['strArea']}\n\n"
            f"Ingredients:\n" + "\n".join(ingredients) +
            f"\n\nInstructions:\n{meal['strInstructions'][:700]}..."
        )
    except requests.RequestException as e:
        return f"Error fetching recipe: {str(e)}"


# ─────────────────────────────────────────────────────────────
# TOOL 2: Browse meals by food category
# ─────────────────────────────────────────────────────────────
@tool
def get_meals_by_category(category: str) -> str:
    """Get meal ideas by category.
    Available: Beef, Chicken, Dessert, Lamb, Pasta, Pork,
    Seafood, Side, Starter, Vegan, Vegetarian, Breakfast, Goat"""
    url = f"https://www.themealdb.com/api/json/v1/1/filter.php?c={category}"
    try:
        resp = requests.get(url, timeout=10)
        data = resp.json()

        if not data.get('meals'):
            return f"No meals found for category '{category}'. Check the category name."

        meals = [m['strMeal'] for m in data['meals'][:6]]
        return f"Popular {category} dishes:\n" + "\n".join(f"  • {m}" for m in meals)
    except requests.RequestException as e:
        return f"Error fetching category: {str(e)}"


# ─────────────────────────────────────────────────────────────
# AGENT SETUP — Amazon Nova 2 Lite via Amazon Bedrock
# ─────────────────────────────────────────────────────────────
model_id = "global.amazon.nova-2-lite-v1:0"
model = BedrockModel(model_id=model_id)

agent = Agent(
    model=model,
    tools=[search_recipe, get_meals_by_category, calculator],
    system_prompt="""You are a warm, friendly personal meal planner and recipe assistant.
You help users discover delicious recipes, suggest meals based on their dietary needs,
and can scale recipes for different serving sizes using the calculator tool.
Always be respectful of dietary restrictions and allergies.
Suggest alternatives when a dish does not fit the user's needs."""
)


def meal_planner(payload: dict) -> str:
    user_input = payload.get("prompt")
    response = agent(user_input)
    return response.message['content'][0]['text']


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("payload", type=str)
    args = parser.parse_args()
    result = meal_planner(json.loads(args.payload))
    print(result)

### Test the Local Agent

Let's test a few queries to confirm everything works before adding memory and streaming.

In [ ]:
# Test 1: Search for a specific recipe
!python meal_planner.py '{"prompt": "Find me a recipe for pasta carbonara"}'

In [ ]:
# Test 2: Browse by category
!python meal_planner.py '{"prompt": "What vegetarian dishes are available?"}'

In [ ]:
# Test 3: Scale a recipe using the calculator
!python meal_planner.py '{"prompt": "Find a pasta recipe and tell me how much of each ingredient I need for 8 people instead of 4"}'

---
## Section 3 — The Memory Problem

The agent works great, but it has a **critical limitation**: it forgets everything between sessions.

### The Problem Illustrated

```
 Session 1 (Monday)
 ─────────────────────────────────────────────
 User:  "I am vegetarian and allergic to nuts."
 Agent: "Got it! Here are some vegetarian, nut-free dishes..."

 Session 2 (Tuesday) — NEW conversation
 ─────────────────────────────────────────────
 User:  "Suggest something for dinner."
 Agent: "How about Chicken Tikka Masala with cashews?"   ← WRONG! Agent forgot!
```

Run the cells below to see this problem in action.

In [ ]:
# Demonstrating the memory problem
# These two calls simulate two separate user sessions

import json
import sys
sys.path.insert(0, '.')
from meal_planner import meal_planner, agent

print("=" * 60)
print("SESSION 1 — User shares dietary restriction")
print("=" * 60)
response_session_1 = meal_planner({"prompt": "I am vegetarian and I am allergic to nuts."})
print(response_session_1)

# Simulate new session: reset agent conversation history
agent.messages = []

print("\n" + "=" * 60)
print("SESSION 2 — New conversation (agent has no memory)")
print("=" * 60)
response_session_2 = meal_planner({"prompt": "Suggest something for dinner tonight."})
print(response_session_2)

print("\n⚠️  Notice: The agent does NOT remember the vegetarian/nut-allergy restriction!")

---
## Section 4 — Add AgentCore Memory

**AgentCore Memory** is a fully managed memory service that lets your agent:
- Store what it learns about users (dietary restrictions, preferences, past meals)
- Retrieve relevant facts at the start of every new conversation
- Keep memory isolated per user (user_id scoping)

### How It Works

```
 Every conversation:

 START  →  Retrieve memories for this user
               ↓
           Add memories to system prompt context
               ↓
           Run agent with enriched context
               ↓
           Save new facts learned from this conversation
           → END
```

### Step 4.1 — Create a Memory Store

Run this **once**. It creates a named memory store in AgentCore. Save the `memory_id` — you will use it throughout.

> **Note:** AgentCore Memory is part of the `bedrock-agentcore-control` client for management
> and `bedrock-agentcore-memory` for read/write operations.

In [ ]:
import boto3
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name

# Control plane client — used to CREATE and DELETE memory stores
agentcore_control = boto3.client('bedrock-agentcore-control', region_name=region)

# Create the memory store (run only once)
# Name rules: [a-zA-Z][a-zA-Z0-9_]{0,47}  — letters, digits, underscores only, no hyphens
# eventExpiryDuration : integer number of days to retain raw conversation events
# memoryStrategies    : SEMANTIC strategy automatically extracts facts/preferences from conversations
memory_response = agentcore_control.create_memory(
    name='mealPlannerMemory',
    description='Stores dietary preferences and meal history per user',
    eventExpiryDuration=30,
    memoryStrategies=[
        {
            'semanticMemoryStrategy': {
                'name': 'mealPlannerSemantic',
                'description': 'Extracts dietary preferences, allergies, and meal choices'
            }
        }
    ]
)

memory_id = memory_response['memory']['id']
print(f"✅ Memory Store created!")
print(f"   Memory ID : {memory_id}")
print(f"   Region    : {region}")
print(f"\n📌 Save this memory_id — it is used in every step below.")

### Step 4.2 — Define Memory Helper Functions

Two simple functions using the `MemorySessionManager` from the `bedrock_agentcore` SDK:
- `save_to_memory()` — stores a conversation turn after the agent responds
- `retrieve_from_memory()` — fetches relevant past facts before the agent responds

> **How it works internally:**
> - `save_to_memory` → `session.add_turns()` → AgentCore extracts facts into **long-term memory** (runs in background ~30s)
> - `retrieve_from_memory` → `session.search_long_term_memories()` → semantic search over stored facts

In [ ]:
from bedrock_agentcore.memory import MemorySessionManager
from bedrock_agentcore.memory.constants import ConversationalMessage, MessageRole

# MemorySessionManager — the recommended SDK for AgentCore Memory read/write
memory_session_mgr = MemorySessionManager(memory_id=memory_id, region_name=region)


def save_to_memory(user_id: str, session_id: str, user_message: str, agent_response: str):
    """
    Save a conversation turn to AgentCore Memory.
    AgentCore automatically extracts facts from the conversation into long-term memory (~30s delay).
    """
    session = memory_session_mgr.create_memory_session(
        actor_id=user_id,
        session_id=session_id
    )
    session.add_turns(messages=[
        ConversationalMessage(user_message,   MessageRole.USER),
        ConversationalMessage(agent_response, MessageRole.ASSISTANT)
    ])
    print(f"✅ Saved to memory | user: {user_id} | session: {session_id}")


def retrieve_from_memory(user_id: str, session_id: str, query: str) -> str:
    """
    Retrieve relevant past facts from AgentCore Memory via semantic search.
    Returns a formatted string to inject into the agent's system prompt.
    """
    session = memory_session_mgr.create_memory_session(
        actor_id=user_id,
        session_id=session_id
    )
    try:
        records = session.search_long_term_memories(
            query=query,
            namespace_path="/",
            top_k=5
        )
        if not records:
            return ""
        facts = []
        for r in records:
            if isinstance(r, dict):
                content = r.get('content', r.get('text', ''))
                if isinstance(content, dict):
                    content = content.get('text', str(content))
                facts.append(str(content))
            else:
                facts.append(str(r))
        return "\n".join(f"  - {f}" for f in facts if f)
    except Exception as e:
        print(f"Memory retrieve note: {e}")
        return ""


print("✅ Memory helper functions defined.")

### Step 4.3 — Session 1: User Shares Preferences

The user tells the agent their dietary restrictions. After the response, we **save this to AgentCore Memory**.

In [ ]:
from meal_planner import meal_planner, agent, search_recipe, get_meals_by_category

user_id    = "student_001"    # Unique identifier per user
session_id = "session_monday" # Unique identifier per conversation

session1_prompt = "I am vegetarian and I am allergic to nuts. What pasta can you suggest?"

print("=" * 60)
print("SESSION 1")
print("=" * 60)
print(f"User: {session1_prompt}\n")

# Run the agent
session1_response = meal_planner({"prompt": session1_prompt})
print(f"Agent: {session1_response}")

# Save this conversation — AgentCore extracts facts in the background
print("\n" + "-" * 60)
save_to_memory(user_id, session_id, session1_prompt, session1_response)
print("💾 AgentCore Memory will extract: user is vegetarian, user is allergic to nuts.")
print("⏳ Wait ~30 seconds before running Session 2 for long-term extraction to complete.")

### Step 4.4 — Session 2: Agent Uses Stored Memory

A **completely new conversation** — the user says nothing about their restrictions.
But the agent retrieves past preferences from AgentCore Memory and applies them automatically.

In [ ]:
from strands import Agent
from strands.models import BedrockModel

# Reset agent state to simulate a brand new conversation
agent.messages = []

user_id      = "student_001"
session_id_2 = "session_tuesday"  # Different session ID = new conversation

session2_prompt = "What should I cook for dinner tonight?"

print("=" * 60)
print("SESSION 2 — New conversation (agent retrieves memory)")
print("=" * 60)
print(f"User: {session2_prompt}\n")

# Step 1: Retrieve past preferences from AgentCore Memory
print("📚 Searching AgentCore Memory for relevant preferences...")
past_preferences = retrieve_from_memory(user_id, session_id_2, session2_prompt)

BASE_SYSTEM_PROMPT = """You are a warm, friendly personal meal planner and recipe assistant.
You help users discover delicious recipes, suggest meals based on their dietary needs,
and can scale recipes using the calculator tool.
Always be respectful of dietary restrictions and allergies."""

# Step 2: Inject memory into the system prompt
if past_preferences:
    print("✅ Retrieved from memory:")
    print(past_preferences)
    print()
    enriched_prompt = BASE_SYSTEM_PROMPT + f"\n\nIMPORTANT — What I know about this user:\n{past_preferences}"
else:
    print("📭 No long-term memories found yet.")
    print("   ↳ Tip: If Session 1 just ran, wait ~30s for extraction to finish, then re-run this cell.\n")
    enriched_prompt = BASE_SYSTEM_PROMPT

# Step 3: Create memory-aware agent with Amazon Nova 2 Lite
model = BedrockModel(model_id="global.amazon.nova-2-lite-v1:0")
memory_agent = Agent(
    model=model,
    tools=[search_recipe, get_meals_by_category],
    system_prompt=enriched_prompt
)

# Step 4: Run the agent with full context
response_s2   = memory_agent(session2_prompt)
result_s2     = response_s2.message['content'][0]['text']
print(f"Agent: {result_s2}")

# Step 5: Save this session too
save_to_memory(user_id, session_id_2, session2_prompt, result_s2)

print("\n✅ The agent respected the nut allergy and vegetarian preference — without being told again!")

---
## Section 5 — Add Streaming

### What is Streaming?

Without streaming, the user waits for the **entire response** to be generated before seeing anything.
With streaming, tokens arrive **one by one** — the user sees output immediately.

```
 Without Streaming:
 User asks → [........5 seconds........] → Full response appears at once

 With Streaming:
 User asks → Here... is... a... great... recipe... (tokens arrive live)
```

### How Strands Handles Streaming

Good news: **Strands already streams to the console by default!**

When you call `agent(prompt)`, Strands uses a callback handler that prints tokens
as they are generated. This is why you saw tool calls like `Tool #1: search_recipe`
printed during local testing.

When deployed to AgentCore Runtime, streaming is delivered to the client
via **SSE (Server-Sent Events)** — a standard web streaming format.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Local Streaming Demo
# Strands streams token-by-token to console by default
# ─────────────────────────────────────────────────────────────
from strands import Agent
from strands.models import BedrockModel
from meal_planner import search_recipe, get_meals_by_category

# Amazon Nova 2 Lite — fast, cost-effective, 1M context window
model = BedrockModel(model_id="global.amazon.nova-2-lite-v1:0")

streaming_agent = Agent(
    model=model,
    tools=[search_recipe, get_meals_by_category],
    system_prompt="You are a friendly meal planner assistant."
)

print("🔄 Streaming response — tokens appear as they are generated:")
print("─" * 60)

# The agent streams to console automatically via Strands' default callback handler
response = streaming_agent(
    "Suggest a complete Italian dinner menu: starter, main course, and dessert with recipe names."
)

print("\n" + "─" * 60)
print("\n✅ Streaming complete. The full response was assembled token by token.")

### Streaming in AgentCore Runtime

When the agent is **deployed to AgentCore**, the invoke response comes back as an
**EventStream**. You parse it chunk by chunk for real-time output.

We will show this live in **Section 6** after deployment.

Here is the pattern you will use:

```python
# boto3 invoke with EventStream parsing
boto3_response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": "...", "user_id": "student_001"})
)

# Stream the response chunks as they arrive
print("🔄 Streaming from AgentCore Runtime:")
for event in boto3_response["response"]:
    chunk = json.loads(event.decode("utf-8"))
    print(chunk, end='', flush=True)
```

---
## Section 6 — Deploy Everything to AgentCore Runtime

Now we combine all three features into a single deployable agent:
- TheMealDB recipe tools
- AgentCore Memory (retrieve before → save after)
- Streaming via AgentCore Runtime

### Step 6.1 — Write the Full Agent File

In [ ]:
%%writefile strands_meal_planner.py
from strands import Agent, tool
from strands_tools import calculator
from strands.models import BedrockModel
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from bedrock_agentcore.memory.integrations.strands.session_manager import AgentCoreMemorySessionManager
from bedrock_agentcore.memory.integrations.strands.config import AgentCoreMemoryConfig, RetrievalConfig
import requests
import boto3
import os

app = BedrockAgentCoreApp()

# ─────────────────────────────────────────────────────────────
# CONFIGURATION
# MEMORY_ID is injected as an env var via runtime.configure()
# ─────────────────────────────────────────────────────────────
region    = boto3.Session().region_name
MEMORY_ID = os.environ.get("MEMORY_ID", "")


# ─────────────────────────────────────────────────────────────
# TOOLS
# ─────────────────────────────────────────────────────────────
@tool
def search_recipe(dish: str) -> str:
    """Search for a recipe by dish name using TheMealDB API"""
    url = f"https://www.themealdb.com/api/json/v1/1/search.php?s={dish}"
    try:
        resp = requests.get(url, timeout=10)
        data = resp.json()
        if not data.get('meals'):
            return f"No recipe found for '{dish}'."
        meal = data['meals'][0]
        ingredients = []
        for i in range(1, 21):
            ing     = meal.get(f'strIngredient{i}', '').strip()
            measure = meal.get(f'strMeasure{i}', '').strip()
            if ing:
                ingredients.append(f"  - {measure} {ing}".strip())
        return (
            f"Recipe: {meal['strMeal']}\n"
            f"Category: {meal['strCategory']}  |  Cuisine: {meal['strArea']}\n\n"
            f"Ingredients:\n" + "\n".join(ingredients) +
            f"\n\nInstructions:\n{meal['strInstructions'][:700]}..."
        )
    except Exception as e:
        return f"Error: {str(e)}"


@tool
def get_meals_by_category(category: str) -> str:
    """Get meal ideas by category.
    Categories: Beef, Chicken, Dessert, Lamb, Pasta, Pork,
    Seafood, Vegan, Vegetarian, Breakfast, Starter"""
    url = f"https://www.themealdb.com/api/json/v1/1/filter.php?c={category}"
    try:
        resp = requests.get(url, timeout=10)
        data = resp.json()
        if not data.get('meals'):
            return f"No meals found for '{category}'."
        meals = [m['strMeal'] for m in data['meals'][:6]]
        return f"Popular {category} dishes:\n" + "\n".join(f"  • {m}" for m in meals)
    except Exception as e:
        return f"Error: {str(e)}"


# ─────────────────────────────────────────────────────────────
# MODEL — Amazon Nova 2 Lite (1M context, fast & cost-effective)
# ─────────────────────────────────────────────────────────────
BASE_SYSTEM_PROMPT = """You are a warm, friendly personal meal planner and recipe assistant.
You help users discover delicious recipes, suggest meals based on their dietary needs,
and can scale recipes for different serving sizes using the calculator tool.
Always be respectful of dietary restrictions and allergies.
Suggest alternatives when a dish does not fit the user's needs."""

model = BedrockModel(model_id="global.amazon.nova-2-lite-v1:0")


# ─────────────────────────────────────────────────────────────
# AGENTCORE ENTRYPOINT
# Uses native Strands + AgentCore Memory integration via session_manager.
# session_manager automatically:
#   • Retrieves relevant memories BEFORE the agent responds
#   • Saves the conversation turn AFTER the agent responds
# ─────────────────────────────────────────────────────────────
@app.entrypoint
async def meal_planner_agent(payload, context):
    user_id    = payload.get("user_id", "default_user")
    user_input = payload.get("prompt")
    session_id = getattr(context, 'session_id', 'default-session')

    print(f"User ID    : {user_id}")
    print(f"Session ID : {session_id}")
    print(f"User Input : {user_input}")

    # Wire AgentCore Memory to the Strands agent via session_manager
    session_manager = None
    if MEMORY_ID:
        session_manager = AgentCoreMemorySessionManager(
            AgentCoreMemoryConfig(
                memory_id=MEMORY_ID,
                session_id=session_id,
                actor_id=user_id,
                retrieval_config={
                    "/": RetrievalConfig(top_k=5, relevance_score=0.5)
                }
            ),
            region
        )

    agent = Agent(
        model=model,
        tools=[search_recipe, get_meals_by_category, calculator],
        session_manager=session_manager,   # Memory handled automatically
        system_prompt=BASE_SYSTEM_PROMPT
    )

    response = agent(user_input)
    return response.message['content'][0]['text']


if __name__ == "__main__":
    app.run()

### Step 6.2 — Configure the Deployment

This generates the `Dockerfile` and sets up the ECR repository and IAM execution role.
We also pass `MEMORY_ID` as an environment variable so the deployed agent can access it.

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()
agent_name = "meal_planner_agent"

response = agentcore_runtime.configure(
    entrypoint="strands_meal_planner.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
    environment_variables={
        "MEMORY_ID": memory_id   # Pass memory store ID to the deployed agent
    }
)

print("\n✅ Configuration complete!")
print(f"   Agent name : {agent_name}")
print(f"   Region     : {response.region}")
print(f"   Account    : {response.account_id}")
response

### Step 6.3 — Launch the Agent

This triggers **AWS CodeBuild** to build the ARM64 Docker image in the cloud
and deploy it to AgentCore Runtime. No local Docker required.

This takes **2–5 minutes**.

In [ ]:
launch_result = agentcore_runtime.launch()
print(f"\n✅ Agent launched!")
print(f"   ARN: {launch_result.agent_arn}")

### Step 6.4 — Wait for the Endpoint to be READY

In [ ]:
import time

status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_statuses = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']

while status not in end_statuses:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(f"Status: {status}")

print(f"\nFinal status: {status}")
if status == 'READY':
    print("✅ Agent is live and ready to receive requests!")
else:
    print("❌ Deployment failed. Check CloudWatch logs for details.")

### Step 6.5 — Invoke the Agent (SDK)

Use the starter toolkit to send a request. The `user_id` tells the agent which
memory profile to load.

In [ ]:
# Session 1 — User shares dietary info
invoke_response = agentcore_runtime.invoke({
    "prompt": "I am vegetarian and allergic to nuts. Suggest a pasta dish.",
    "user_id": "student_001"
})

from IPython.display import Markdown, display
response_text = invoke_response['response'][0]
display(Markdown(response_text))

In [ ]:
# Session 2 — New conversation, agent remembers preferences
invoke_response_2 = agentcore_runtime.invoke({
    "prompt": "What should I cook for dinner tonight?",
    "user_id": "student_001"
})

response_text_2 = invoke_response_2['response'][0]
display(Markdown(response_text_2))
print("\n✅ Notice the agent respects nut-allergy and vegetarian preference without being told again!")

### Step 6.6 — Invoke with Streaming (boto3)

Use the raw boto3 client and parse the **EventStream** for real-time streaming output.

In [ ]:
import boto3
import json

agent_arn = launch_result.agent_arn

agentcore_client = boto3.client('bedrock-agentcore', region_name=region)

boto3_response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier="DEFAULT",
    payload=json.dumps({
        "prompt": "Create a full Italian dinner menu for 4 people: starter, main, and dessert.",
        "user_id": "student_001"
    })
)

print("🔄 Streaming response from AgentCore Runtime:\n")
print("─" * 60)

if "text/event-stream" in boto3_response.get("contentType", ""):
    # SSE streaming — tokens arrive line by line
    content_parts = []
    for line in boto3_response["response"].iter_lines(chunk_size=1):
        if line:
            decoded = line.decode("utf-8")
            if decoded.startswith("data: "):
                token = decoded[6:]  # Strip the "data: " prefix
                print(token, end='', flush=True)
                content_parts.append(token)
    full_response = "".join(content_parts)
else:
    # EventStream (JSON chunks)
    events = []
    for event in boto3_response.get("response", []):
        events.append(event)
    full_response = json.loads(events[0].decode("utf-8")) if events else ""
    print(full_response)

print("\n" + "─" * 60)
print("\n✅ Streaming complete! The response was delivered token-by-token.")

---
## Section 7 — Cleanup

Delete the AgentCore Runtime, ECR repository, and Memory Store to avoid ongoing charges.

In [ ]:
# Delete AgentCore Runtime and ECR repository
agentcore_control_client = boto3.client('bedrock-agentcore-control', region_name=region)
ecr_client = boto3.client('ecr', region_name=region)

agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=launch_result.agent_id
)
print("✅ AgentCore Runtime deleted.")

ecr_client.delete_repository(
    repositoryName=launch_result.ecr_uri.split('/')[1],
    force=True
)
print("✅ ECR repository deleted.")

In [ ]:
# Delete the AgentCore Memory Store
agentcore_control.delete_memory(
    memoryId=memory_id
)
print(f"✅ AgentCore Memory Store deleted: {memory_id}")

---
# Congratulations!

You built and deployed a production-ready AI agent with:

| Feature | What You Learned |
|:---|:---|
| **Strands Agent** | Build tool-calling agents with clean Python decorators |
| **Real API Integration** | Connect to TheMealDB for live recipe data |
| **AgentCore Memory** | Give your agent persistent, cross-session user memory |
| **Streaming** | Deliver responses token-by-token for better UX |
| **AgentCore Runtime** | Deploy serverlessly with CodeBuild + ECR + ARM64 containers |

### What to Try Next
- Add a **new tool**: fetch nutritional info for a recipe ingredient
- Add **multi-user support**: test with `user_id: "student_002"` and see isolated memories
- Connect the agent to a **frontend** using the boto3 `invoke_agent_runtime` call